# T1 and FreeSurfer Validation

Validation focuses on image properties, FreeSurfer output consistency, cortical thickness distributions, and cortical parcellation rather than direct vertex-wise correspondence.

# 0. Import

In [25]:
from pathlib import Path

import numpy as np
import pandas as pd

from nibabel.freesurfer.io import (
    read_annot,
    read_geometry,
    read_morph_data,
)


SUBJECTS = [
    "D0007",
    "S00775",
]

HEMISPHERES = [
    "lh",
    "rh",
]

DATA_ROOT = Path(
    "/mnt/newStor/paros/paros_WORK/xintong/data/column_sample_data"
)

NEW_PIPELINE_ROOT = Path(
    "/mnt/newStor/paros/paros_WORK/xintong/test/new_pipeline"
)

RESULTS_ROOT = Path(
    "/mnt/newStor/paros/paros_WORK/xintong/test/comparison/t1_validation"
)

RESULTS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

print("Subjects:", SUBJECTS)
print("Results:", RESULTS_ROOT)

Subjects: ['D0007', 'S00775']
Results: /mnt/newStor/paros/paros_WORK/xintong/test/comparison/t1_validation


## a. Create paths and validation summary

In [26]:
def get_freesurfer_paths(subject):
    """
    Return old and new FreeSurfer directories.
    """

    old_fs = (
        DATA_ROOT
        / subject
        / subject
    )

    new_fs = (
        NEW_PIPELINE_ROOT
        / subject
        / "freesurfer"
        / subject
    )

    return {
        "old": old_fs,
        "new": new_fs,
        "results": RESULTS_ROOT / subject,
    }


FILES = {
    subject: get_freesurfer_paths(subject)
    for subject in SUBJECTS
}


for subject, paths in FILES.items():
    print()
    print(subject)
    print("Old:", paths["old"])
    print("New:", paths["new"])


D0007
Old: /mnt/newStor/paros/paros_WORK/xintong/data/column_sample_data/D0007/D0007
New: /mnt/newStor/paros/paros_WORK/xintong/test/new_pipeline/D0007/freesurfer/D0007

S00775
Old: /mnt/newStor/paros/paros_WORK/xintong/data/column_sample_data/S00775/S00775
New: /mnt/newStor/paros/paros_WORK/xintong/test/new_pipeline/S00775/freesurfer/S00775


In [27]:
from pathlib import Path

import numpy as np
from nibabel.freesurfer.io import read_geometry


OLD_FS = Path(
    "/mnt/newStor/paros/paros_WORK/xintong/data/"
    "column_sample_data/S00775/S00775"
)

NEW_FS = Path(
    "/mnt/newStor/paros/paros_WORK/xintong/test/"
    "new_pipeline/S00775/freesurfer/S00775"
)

print("Old FreeSurfer:", OLD_FS)
print("New FreeSurfer:", NEW_FS)

Old FreeSurfer: /mnt/newStor/paros/paros_WORK/xintong/data/column_sample_data/S00775/S00775
New FreeSurfer: /mnt/newStor/paros/paros_WORK/xintong/test/new_pipeline/S00775/freesurfer/S00775


In [28]:
required_files = [
    "surf/lh.white",
    "surf/rh.white",
    "surf/lh.pial",
    "surf/rh.pial",
    "surf/lh.thickness",
    "surf/rh.thickness",
    "label/lh.aparc.annot",
    "label/rh.aparc.annot",
]


def check_required_files(subject, paths):
    """
    Check required FreeSurfer outputs.
    """

    print()
    print(f"Subject: {subject}")

    all_found = True

    for relative_path in required_files:
        old_path = paths["old"] / relative_path
        new_path = paths["new"] / relative_path

        old_exists = old_path.is_file()
        new_exists = new_path.is_file()

        if not old_exists or not new_exists:
            all_found = False

        print(relative_path)
        print(f"  old: {old_exists}")
        print(f"  new: {new_exists}")

    return all_found


FILE_STATUS = {}

for subject in SUBJECTS:
    FILE_STATUS[subject] = check_required_files(
        subject,
        FILES[subject],
    )


Subject: D0007
surf/lh.white
  old: True
  new: True
surf/rh.white
  old: True
  new: True
surf/lh.pial
  old: True
  new: True
surf/rh.pial
  old: True
  new: True
surf/lh.thickness
  old: True
  new: True
surf/rh.thickness
  old: True
  new: True
label/lh.aparc.annot
  old: True
  new: True
label/rh.aparc.annot
  old: True
  new: True

Subject: S00775
surf/lh.white
  old: True
  new: True
surf/rh.white
  old: True
  new: True
surf/lh.pial
  old: True
  new: True
surf/rh.pial
  old: True
  new: True
surf/lh.thickness
  old: True
  new: True
surf/rh.thickness
  old: True
  new: True
label/lh.aparc.annot
  old: True
  new: True
label/rh.aparc.annot
  old: True
  new: True


# Surface topology comparison

In [29]:
def compare_surface_topology(
    subject,
    hemisphere,
    old_fs,
    new_fs,
):
    """
    Compare white-surface mesh topology.
    """

    old_path = (
        old_fs
        / "surf"
        / f"{hemisphere}.white"
    )

    new_path = (
        new_fs
        / "surf"
        / f"{hemisphere}.white"
    )

    old_vertices, old_faces = read_geometry(old_path)
    new_vertices, new_faces = read_geometry(new_path)

    same_vertex_count = (
        old_vertices.shape[0]
        == new_vertices.shape[0]
    )

    same_face_count = (
        old_faces.shape[0]
        == new_faces.shape[0]
    )

    same_faces = False

    if old_faces.shape == new_faces.shape:
        same_faces = np.array_equal(
            old_faces,
            new_faces,
        )

    result = {
        "subject": subject,
        "hemisphere": hemisphere,
        "old_vertices": old_vertices.shape[0],
        "new_vertices": new_vertices.shape[0],
        "old_faces": old_faces.shape[0],
        "new_faces": new_faces.shape[0],
        "same_vertex_count": same_vertex_count,
        "same_face_count": same_face_count,
        "same_faces": same_faces,
    }

    return result

In [30]:
topology_results = []

for subject in SUBJECTS:
    paths = FILES[subject]

    for hemisphere in HEMISPHERES:
        result = compare_surface_topology(
            subject,
            hemisphere,
            paths["old"],
            paths["new"],
        )

        topology_results.append(result)


topology_df = pd.DataFrame(
    topology_results
)

display(topology_df)

,subject,hemisphere,old_vertices,new_vertices,old_faces,new_faces,same_vertex_count,same_face_count,same_faces
0,D0007,lh,118922,120251,237840,240498,False,False,False
1,D0007,rh,118767,119685,237530,239366,False,False,False
2,S00775,lh,121365,125751,242726,251498,False,False,False
3,S00775,rh,123599,128129,247194,256254,False,False,False


In [31]:
def check_freesurfer_integrity(
    subject,
    hemisphere,
    fs_dir,
    source,
):
    """
    Check internal consistency of one FreeSurfer reconstruction.
    """

    white_path = (
        fs_dir
        / "surf"
        / f"{hemisphere}.white"
    )

    pial_path = (
        fs_dir
        / "surf"
        / f"{hemisphere}.pial"
    )

    thickness_path = (
        fs_dir
        / "surf"
        / f"{hemisphere}.thickness"
    )

    annot_path = (
        fs_dir
        / "label"
        / f"{hemisphere}.aparc.annot"
    )

    white_vertices, white_faces = read_geometry(
        white_path
    )

    pial_vertices, pial_faces = read_geometry(
        pial_path
    )

    thickness = read_morph_data(
        thickness_path
    )

    labels, _, names = read_annot(
        annot_path
    )

    return {
        "subject": subject,
        "hemisphere": hemisphere,
        "source": source,

        "white_vertices": len(white_vertices),
        "pial_vertices": len(pial_vertices),

        "white_faces": len(white_faces),
        "pial_faces": len(pial_faces),

        "thickness_values": len(thickness),
        "annotation_values": len(labels),
        "regions": len(names),

        "white_pial_vertex_match": (
            len(white_vertices)
            == len(pial_vertices)
        ),

        "thickness_vertex_match": (
            len(thickness)
            == len(white_vertices)
        ),

        "annotation_vertex_match": (
            len(labels)
            == len(white_vertices)
        ),

        "white_pial_topology_match": (
            white_faces.shape == pial_faces.shape
            and np.array_equal(
                white_faces,
                pial_faces,
            )
        ),
    }

In [32]:
integrity_results = []

for subject in SUBJECTS:
    paths = FILES[subject]

    for hemisphere in HEMISPHERES:

        for source in [
            "old",
            "new",
        ]:

            result = check_freesurfer_integrity(
                subject,
                hemisphere,
                paths[source],
                source,
            )

            integrity_results.append(
                result
            )


integrity_df = pd.DataFrame(
    integrity_results
)

display(integrity_df)

,subject,hemisphere,source,white_vertices,pial_vertices,white_faces,pial_faces,thickness_values,annotation_values,regions,white_pial_vertex_match,thickness_vertex_match,annotation_vertex_match,white_pial_topology_match
0,D0007,lh,old,118922,118922,237840,237840,118922,118922,36,True,True,True,True
1,D0007,lh,new,120251,120251,240498,240498,120251,120251,36,True,True,True,True
2,D0007,rh,old,118767,118767,237530,237530,118767,118767,36,True,True,True,True
3,D0007,rh,new,119685,119685,239366,239366,119685,119685,36,True,True,True,True
4,S00775,lh,old,121365,121365,242726,242726,121365,121365,36,True,True,True,True
5,S00775,lh,new,125751,125751,251498,251498,125751,125751,36,True,True,True,True
6,S00775,rh,old,123599,123599,247194,247194,123599,123599,36,True,True,True,True
7,S00775,rh,new,128129,128129,256254,256254,128129,128129,36,True,True,True,True


# 3. Region_distribution

In [33]:
def get_region_distribution(
    subject,
    hemisphere,
    fs_dir,
    source,
):
    """
    Calculate aparc region vertex counts and proportions.
    """

    annot_path = (
        fs_dir
        / "label"
        / f"{hemisphere}.aparc.annot"
    )

    labels, _, names = read_annot(
        annot_path
    )

    rows = []

    total_vertices = len(labels)

    for label_id, name in enumerate(names):
        if isinstance(name, bytes):
            name = name.decode("utf-8")

        mask = labels == label_id
        n_vertices = int(np.sum(mask))

        rows.append({
            "subject": subject,
            "hemisphere": hemisphere,
            "source": source,
            "region": name,
            "n_vertices": n_vertices,
            "proportion": (
                n_vertices
                / total_vertices
            ),
        })

    return pd.DataFrame(rows)

In [34]:
region_distribution_tables = []

for subject in SUBJECTS:
    paths = FILES[subject]

    for hemisphere in HEMISPHERES:
        for source in ["old", "new"]:

            df = get_region_distribution(
                subject,
                hemisphere,
                paths[source],
                source,
            )

            region_distribution_tables.append(df)


region_distribution_df = pd.concat(
    region_distribution_tables,
    ignore_index=True,
)

display(region_distribution_df.head(20))

,subject,hemisphere,source,region,n_vertices,proportion
0,D0007,lh,old,unknown,0,0.000000
1,D0007,lh,old,bankssts,1201,0.010099
2,D0007,lh,old,caudalanteriorcingulate,800,0.006727
3,D0007,lh,old,caudalmiddlefrontal,3180,0.026740
4,D0007,lh,old,corpuscallosum,0,0.000000
5,D0007,lh,old,cuneus,2378,0.019996
6,D0007,lh,old,entorhinal,569,0.004785
7,D0007,lh,old,fusiform,3362,0.028271
8,D0007,lh,old,inferiorparietal,5430,0.045660
9,D0007,lh,old,inferiortemporal,3548,0.029835


In [35]:
region_distribution_comparison = (
    region_distribution_df
    .pivot_table(
        index=[
            "subject",
            "hemisphere",
            "region",
        ],
        columns="source",
        values=[
            "n_vertices",
            "proportion",
        ],
    )
    .reset_index()
)


region_distribution_comparison.columns = [
    "_".join(
        col
        if isinstance(col, tuple)
        else [col]
    ).strip("_")
    for col in region_distribution_comparison.columns
]


region_distribution_comparison[
    "proportion_difference"
] = (
    region_distribution_comparison[
        "proportion_new"
    ]
    -
    region_distribution_comparison[
        "proportion_old"
    ]
)


region_distribution_comparison[
    "abs_proportion_difference"
] = np.abs(
    region_distribution_comparison[
        "proportion_difference"
    ]
)


display(
    region_distribution_comparison
)

,subject,hemisphere,region,n_vertices_new,n_vertices_old,proportion_new,proportion_old,proportion_difference,abs_proportion_difference
0,D0007,lh,bankssts,1201.0,1201.0,0.009987,0.010099,-0.000112,0.000112
1,D0007,lh,caudalanteriorcingulate,820.0,800.0,0.006819,0.006727,0.000092,0.000092
2,D0007,lh,caudalmiddlefrontal,3135.0,3180.0,0.026070,0.026740,-0.000670,0.000670
3,D0007,lh,corpuscallosum,0.0,0.0,0.000000,0.000000,0.000000,0.000000
4,D0007,lh,cuneus,2292.0,2378.0,0.019060,0.019996,-0.000936,0.000936
...,...,...,...,...,...,...,...,...,...
139,S00775,rh,superiortemporal,5794.0,5752.0,0.045220,0.046538,-0.001318,0.001318
140,S00775,rh,supramarginal,5525.0,5386.0,0.043121,0.043576,-0.000456,0.000456
141,S00775,rh,temporalpole,536.0,537.0,0.004183,0.004345,-0.000161,0.000161
142,S00775,rh,transversetemporal,670.0,607.0,0.005229,0.004911,0.000318,0.000318


# 4.Thickness

In [36]:
def get_regional_thickness(
    subject,
    hemisphere,
    fs_dir,
    source,
):
    thickness = read_morph_data(
        fs_dir / "surf" / f"{hemisphere}.thickness"
    )

    labels, _, names = read_annot(
        fs_dir / "label" / f"{hemisphere}.aparc.annot"
    )

    rows = []

    for label_id, name in enumerate(names):
        if isinstance(name, bytes):
            name = name.decode("utf-8")

        mask = labels == label_id

        if not np.any(mask):
            continue

        rows.append({
            "subject": subject,
            "hemisphere": hemisphere,
            "region": name,
            "source": source,
            "mean_thickness_mm": np.mean(thickness[mask]),
        })

    return pd.DataFrame(rows)

In [37]:
tables = []

for subject in SUBJECTS:
    for hemisphere in HEMISPHERES:
        for source in ["old", "new"]:

            df = get_regional_thickness(
                subject,
                hemisphere,
                FILES[subject][source],
                source,
            )

            tables.append(df)

regional_thickness_df = pd.concat(
    tables,
    ignore_index=True,
)

display(regional_thickness_df.head())

,subject,hemisphere,region,source,mean_thickness_mm
0,D0007,lh,bankssts,old,2.257646
1,D0007,lh,caudalanteriorcingulate,old,2.330830
2,D0007,lh,caudalmiddlefrontal,old,2.366693
3,D0007,lh,cuneus,old,1.736209
4,D0007,lh,entorhinal,old,2.905538


In [38]:
regional_thickness_comparison = (
    regional_thickness_df
    .pivot_table(
        index=[
            "subject",
            "hemisphere",
            "region",
        ],
        columns="source",
        values="mean_thickness_mm",
    )
    .reset_index()
)

regional_thickness_comparison["difference_mm"] = (
    regional_thickness_comparison["new"]
    - regional_thickness_comparison["old"]
)

regional_thickness_comparison["abs_difference_mm"] = (
    regional_thickness_comparison["difference_mm"].abs()
)

display(regional_thickness_comparison)

source,subject,hemisphere,region,new,old,difference_mm,abs_difference_mm
0,D0007,lh,bankssts,2.242059,2.257646,-0.015587,0.015587
1,D0007,lh,caudalanteriorcingulate,2.357403,2.330830,0.026572,0.026572
2,D0007,lh,caudalmiddlefrontal,2.386657,2.366693,0.019964,0.019964
3,D0007,lh,cuneus,1.761596,1.736209,0.025387,0.025387
4,D0007,lh,entorhinal,2.857460,2.905538,-0.048078,0.048078
...,...,...,...,...,...,...,...
131,S00775,rh,superiorparietal,2.078347,2.041898,0.036449,0.036449
132,S00775,rh,superiortemporal,2.785741,2.769943,0.015798,0.015798
133,S00775,rh,supramarginal,2.531139,2.557139,-0.026000,0.026000
134,S00775,rh,temporalpole,3.217093,3.377079,-0.159986,0.159986


In [39]:
summary = []

for subject in SUBJECTS:

    df = regional_thickness_comparison[
        regional_thickness_comparison["subject"] == subject
    ].dropna(
        subset=["old", "new"]
    )

    difference = (
        df["new"]
        - df["old"]
    )

    summary.append({
        "subject": subject,
        "n_regions": len(df),

        "correlation": np.corrcoef(
            df["old"],
            df["new"],
        )[0, 1],

        "mae_mm": np.mean(
            np.abs(difference)
        ),

        "rmse_mm": np.sqrt(
            np.mean(difference ** 2)
        ),

        "mean_difference_mm": np.mean(
            difference
        ),
    })


thickness_summary_df = pd.DataFrame(summary)

display(thickness_summary_df)

,subject,n_regions,correlation,mae_mm,rmse_mm,mean_difference_mm
0,D0007,68,0.994259,0.031278,0.045099,-0.001257
1,S00775,68,0.990196,0.042121,0.063664,-0.015251
